# P2-A5 — Tool Calling (the foundation of agents)

So far the model only *talks*. Now you let it **act**: you describe functions it's allowed to call, and the model decides *when* to call them and *with what arguments*. You execute the function and hand the result back. This loop — model → tool call → execute → feed result → final answer — is the seed of every agent.

Goal: see the model choose a tool, emit structured arguments, and incorporate a real result into its answer.

In [ ]:
# --- Setup (given) ---
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'

# --- A real tool: looks up an order. (Pretend this hits a database.) ---
def get_order_status(order_id):
    db = {
        'A123': {'status': 'shipped', 'eta_days': 2},
        'B456': {'status': 'processing', 'eta_days': 5},
    }
    return db.get(order_id, {'status': 'not_found'})

# The schema that tells the model the tool exists and how to call it
tools = [{
    'type': 'function',
    'function': {
        'name': 'get_order_status',
        'description': "Look up the status and ETA of a customer's order by its ID.",
        'parameters': {
            'type': 'object',
            'properties': {
                'order_id': {'type': 'string', 'description': 'The order ID, e.g. A123'},
            },
            'required': ['order_id'],
        },
    },
}]

# Map tool name -> the actual python function, so you can dispatch by name
TOOL_REGISTRY = {'get_order_status': get_order_status}
print('tool ready:', list(TOOL_REGISTRY))

## Task 1 — The model decides to call the tool
Send the message *"Where is my order A123?"* with `tools=tools` passed to `client.chat.completions.create(...)`. Inspect the response: instead of text, the model should return a **tool call**. Print the tool name it picked and the arguments (parse them with `json.loads`).
<br>Hint: the reply is `resp.choices[0].message`. Look at `.tool_calls`; each call has `.function.name` and `.function.arguments` (a JSON *string*).

In [ ]:
# TODO: call with tools=, inspect message.tool_calls, print name + parsed args


## Task 2 — Close the loop: execute the tool, feed the result back
This is the heart of tool use. Steps:
1. Take the tool call from Task 1, dispatch it via `TOOL_REGISTRY[name](**args)` to get the real result.
2. Append to your `messages`: **(a)** the assistant message that contained the tool call, then **(b)** a message with `role='tool'`, the matching `tool_call_id`, and the function result (as a string) for `content`.
3. Call the model **again** with the updated messages — now it has the data and will write a natural-language answer.
<br>Goal: end with something like *"Your order A123 has shipped and should arrive in about 2 days."*
<br>Hint: the tool message shape is `{'role': 'tool', 'tool_call_id': tc.id, 'content': json.dumps(result)}`.

In [ ]:
# TODO: execute tool -> append assistant + tool messages -> call again -> print final answer


## Task 3 — The model knows when NOT to use a tool
Send a message that has nothing to do with orders (e.g. *"Hi, what can you help me with?"*) with the same `tools=tools`. Show that the model answers **directly with text** and `message.tool_calls` is empty/None.
<br>Goal: tool use is the model's *choice*, not forced — it routes based on the request.

In [ ]:
# TODO: a non-tool message; show tool_calls is empty and it replies with text


## Task 4 — Explain (in your own words)
1. How is tool calling different from RAG? (In RAG *you* always fetch context; here the model decides.)
2. You now have the building block of an **agent**. In one paragraph, how would you extend this single tool-call loop into an agent that can handle a multi-step task (e.g. *'cancel my most recent order'*)?

> _your answer here_